# Logistic Loss & Cost – Solution

**Short name (GitHub):** `Logistic_Loss_Cost`  
**Lab source:** C1_W3 Lab04 + Lab05  
Work the **Practice Skeleton** first. This notebook is the worked key: loop + vectorized alternates, extra practice, simulation, and audience write-ups.


## Inline cheat-sheet

| Item | Formula / code |
|------|----------------|
| Sigmoid | `1/(1+np.exp(-np.clip(z,-50,50)))` |
| Loss | `- (y*log(f) + (1-y)*log(1-f))` with $f$ clipped |
| Cost | mean of the per-example losses |
| Lab05 check | $J(w=(1,1),b=-3)\approx 0.36687$, $J(b=-4)\approx 0.50368$ |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")



## 1. Data + why squared error fails


In [ ]:
arr = np.loadtxt("data/logistic_loss_1d.csv", delimiter=",", skiprows=1)
x_1d, y_1d = arr[:, 0], arr[:, 1]
print("m =", x_1d.shape[0])
print("labels:", y_1d)

fig, ax = plt.subplots(figsize=(6, 3.3))
ax.scatter(x_1d[y_1d == 0], y_1d[y_1d == 0], c="#1f77b4", s=70, label="y=0 benign")
ax.scatter(x_1d[y_1d == 1], y_1d[y_1d == 1], c="#d62728", marker="x", s=80, label="y=1 malignant")
ax.set_xlabel("Tumor size (cm)"); ax.set_ylabel("label")
ax.set_title("1-D example"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



In [ ]:
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def squared_cost_logistic(x, y, w, b):
    f = sigmoid(w * x + b)
    return float(np.mean((f - y) ** 2))

print("sigmoid(0) =", sigmoid(0.0))
print("J_sq (2.2,-4.4) =", squared_cost_logistic(x_1d, y_1d, 2.2, -4.4))
print("J_sq (0.6,-0.4)  =", squared_cost_logistic(x_1d, y_1d, 0.6, -0.4))

# small grid to show the squared-error surface is not a clean bowl
ws = np.linspace(-1, 5, 25)
bs = np.linspace(-7, 2, 25)
W, B = np.meshgrid(ws, bs)
Jsq = np.vectorize(lambda ww, bb: squared_cost_logistic(x_1d, y_1d, ww, bb))(W, B)
fig, ax = plt.subplots(figsize=(5.2, 3.6))
cs = ax.contourf(W, B, Jsq, levels=16, cmap="viridis")
plt.colorbar(cs, ax=ax, label="J_sq")
ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_title("Squared error + sigmoid")
plt.show()



**Alternate (pure Python loop for squared error)** — same numbers, useful when you are tracing a single example by hand.


In [ ]:
def squared_cost_loop(x, y, w, b):
    m = len(x); acc = 0.0
    for i in range(m):
        f = sigmoid(w * x[i] + b)
        acc += (f - y[i]) ** 2
    return acc / m

print("loop J_sq =", squared_cost_loop(x_1d, y_1d, 2.2, -4.4))



## 2. Logistic loss


In [ ]:
def _clip_f(f):
    return np.clip(np.asarray(f, dtype=float), 1e-15, 1 - 1e-15)

def logistic_loss_piecewise(f, y):
    f = float(_clip_f(f))
    y = float(y)
    if y == 1:
        return -np.log(f)
    return -np.log(1 - f)

def logistic_loss_compact(f, y):
    f = _clip_f(f)
    y = np.asarray(y, dtype=float)
    return -(y * np.log(f) + (1 - y) * np.log(1 - f))

for f, y in [(0.9, 1), (0.1, 0), (0.9, 0), (0.1, 1)]:
    a = logistic_loss_piecewise(f, y)
    b = float(logistic_loss_compact(np.array([f]), np.array([y])))
    print(f"f={f}, y={y}: piecewise={a:.4f}  compact={b:.4f}")



In [ ]:
f = np.linspace(1e-3, 1 - 1e-3, 400)
fig, ax = plt.subplots(figsize=(6.2, 3.5))
ax.plot(f, -np.log(f), label=r"$y=1:\,-\log(f)$")
ax.plot(f, -np.log(1 - f), label=r"$y=0:\,-\log(1-f)$")
ax.set_xlabel(r"$f_{w,b}(x)$"); ax.set_ylabel("loss")
ax.set_title("Logistic loss curves"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



**Alternate:** `np.where` instead of an `if` (handy when $y$ is already an array).


In [ ]:
def logistic_loss_where(f, y):
    f = _clip_f(f); y = np.asarray(y, dtype=float)
    return np.where(y == 1, -np.log(f), -np.log(1 - f))

print(logistic_loss_where([0.9, 0.1, 0.8], [1, 0, 0]))



## 3. Cost — loop and vectorized


In [ ]:
def compute_cost_logistic(X, y, w, b):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        if X.ndim == 1:
            z_i = w * X[i] + b
        else:
            z_i = np.dot(X[i], w) + b
        f_i = float(sigmoid(z_i))
        cost += float(logistic_loss_compact(f_i, y[i]))
    return cost / m

def compute_cost_logistic_vec(X, y, w, b):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    if X.ndim == 1:
        z = w * X + b
    else:
        z = X @ w + b
    f = _clip_f(sigmoid(z))
    return float(np.mean(-(y * np.log(f) + (1 - y) * np.log(1 - f))))

print("1-D J(2.2, -4.4) =", compute_cost_logistic(x_1d, y_1d, 2.2, -4.4))
print("1-D J(0.6, -0.4)  =", compute_cost_logistic(x_1d, y_1d, 0.6, -0.4))
print("loop vs vec close?",
      np.isclose(compute_cost_logistic(x_1d, y_1d, 2.2, -4.4),
                 compute_cost_logistic_vec(x_1d, y_1d, 2.2, -4.4)))



**Alternate #2 — `np.einsum` / broadcasting notes.**  
For 2-D $X$ the score is always `X @ w + b`. You can also write `np.einsum('ij,j->i', X, w) + b`.

**Alternate #3 — sklearn** (if installed):
```python
from sklearn.metrics import log_loss
log_loss(y, f, labels=[0, 1])   # f = predicted probabilities
```
That call is the same mean BCE we implement here (it also clips internally).


## 4. Two-feature lab example


In [ ]:
arr2 = np.loadtxt("data/logistic_loss_2d.csv", delimiter=",", skiprows=1)
X_2d, y_2d = arr2[:, :2], arr2[:, 2]
w_tmp = np.array([1.0, 1.0])
print("J b=-3:", compute_cost_logistic(X_2d, y_2d, w_tmp, -3))
print("J b=-4:", compute_cost_logistic(X_2d, y_2d, w_tmp, -4))
print("vec  b=-3:", compute_cost_logistic_vec(X_2d, y_2d, w_tmp, -3))

x0 = np.linspace(0, 4, 50)
fig, ax = plt.subplots(figsize=(5, 4.2))
ax.scatter(X_2d[y_2d == 0, 0], X_2d[y_2d == 0, 1], c="#1f77b4", s=80, label="y=0")
ax.scatter(X_2d[y_2d == 1, 0], X_2d[y_2d == 1, 1], c="#d62728", marker="x", s=90, label="y=1")
ax.plot(x0, 3 - x0, label="b=-3 (better)")
ax.plot(x0, 4 - x0, ls="--", label="b=-4 (worse)")
ax.set_xlim(0, 4); ax.set_ylim(0, 3.6)
ax.set_xlabel("$x_0$"); ax.set_ylabel("$x_1$")
ax.set_title("Decision boundaries vs cost ranking")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



Expected: $J(b=-3)\approx 0.366867$, $J(b=-4)\approx 0.503681$.  
The magenta line leaves a positive point on the wrong side of the boundary, so average surprise (cost) is higher.


## 5. More practice — worked


In [ ]:
f_tab = np.array([0.95, 0.20, 0.80])
y_tab = np.array([1.0, 0.0, 0.0])
L = logistic_loss_compact(f_tab, y_tab)
print("per-example losses:", L)
print("mean cost:", float(np.mean(L)))
# pair 3 is the expensive one: y=0 but f=0.80



In [ ]:
prac = np.loadtxt("data/logistic_loss_practice.csv", delimiter=",", skiprows=1)
Xp, yp = prac[:, :2], prac[:, 2]
print("practice J(1.0, 0.8, -0.4) =", compute_cost_logistic_vec(Xp, yp, np.array([1.0, 0.8]), -0.4))
print("practice J(0, 0, 0)         =", compute_cost_logistic_vec(Xp, yp, np.array([0.0, 0.0]), 0.0))
print("baseline meaning: w=0,b=0 => f=0.5 for every row => J = -log(0.5) ≈", -np.log(0.5))



In [ ]:
print("raw -log(0) ->", end=" ")
try:
    print(-np.log(0.0))
except Exception as e:
    print(type(e), e)
print("clipped -log(1e-15) =", -np.log(1e-15))
print("Without clipping, a single f=0 or f=1 example yields inf/nan and poisons the whole batch gradient.")



## 6. Simulation — worked


In [ ]:
TRUE_W = np.array([1.4, 1.1])
TRUE_B = -0.2
M_LIST = [20, 40, 80, 160, 320]
NOISE_LIST = [0.00, 0.05, 0.10, 0.20, 0.30]
N_REPS = 20
SEED = 7

def make_dataset(m, noise, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(m, 2))
    p = sigmoid(X @ TRUE_W + TRUE_B)
    y = (rng.random(m) < p).astype(float)
    nflip = int(noise * m)
    if nflip:
        idx = rng.choice(m, size=nflip, replace=False)
        y[idx] = 1 - y[idx]
    return X, y

def mc_cost(ms=None, noise=0.05, m_fixed=None):
    means, stds, xs = [], [], []
    grid = ms if ms is not None else [m_fixed] * len(NOISE_LIST)
    driver = ms if ms is not None else NOISE_LIST
    for k, val in enumerate(driver):
        cs = []
        for r in range(N_REPS):
            if ms is not None:
                X, y = make_dataset(val, noise, SEED + 17 * r + val)
            else:
                X, y = make_dataset(m_fixed, val, SEED + 31 * r + int(100 * val))
            cs.append(compute_cost_logistic_vec(X, y, TRUE_W, TRUE_B))
        means.append(np.mean(cs)); stds.append(np.std(cs)); xs.append(val)
    return np.array(xs), np.array(means), np.array(stds)

mx, mmean, mstd = mc_cost(ms=M_LIST, noise=0.05)
nx, nmean, nstd = mc_cost(ms=None, noise=None, m_fixed=120)

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.5))
axes[0].errorbar(mx, mmean, yerr=mstd, marker="o", capsize=3)
axes[0].set_xlabel("m"); axes[0].set_ylabel("J at true (w,b)")
axes[0].set_title("Cost vs sample size (5% flips)")
axes[0].grid(True, alpha=0.3)
axes[1].errorbar(nx, nmean, yerr=nstd, marker="s", color="#d62728", capsize=3)
axes[1].set_xlabel("label-flip rate"); axes[1].set_ylabel("J at true (w,b)")
axes[1].set_title("Cost vs label noise (m=120)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("sample-size means:", mmean)
print("noise means:", nmean)



In [ ]:
f = np.linspace(0.01, 0.99, 99)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.plot(f, logistic_loss_compact(f, np.ones_like(f)), label="true y=1")
ax.plot(f, logistic_loss_compact(f, np.zeros_like(f)), label="true y=0")
ax.axvline(0.5, color="gray", ls=":", lw=1)
ax.set_xlabel("predicted probability f"); ax.set_ylabel("loss")
ax.set_title("Overconfidence is expensive")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()



## 7. Audience notes — worked examples

These follow the attached audience guides: match data literacy, subject knowledge, and the reader’s job (analyst vs executive vs non-specialist). The data-analysis report structure (headline in the intro, evidence in the body, detail in an appendix) is the same pattern.


In [ ]:
analyst_note = """
Analyst: On the 6-point 2-D set, J(w=(1,1), b=-3)=0.367 and J(b=-4)=0.504.
The ranking matches the geometry: b=-4 pushes the line x0+x1=4 past a positive example.
Use the compact BCE with clipped probabilities; the loop and vectorized forms agree to 1e-12.
Squared error on top of a sigmoid is not a reliable training objective — the (w,b) heatmap is not convex
in the way the soup-bowl of linear regression is. Next step is dJ/dw, dJ/db and gradient descent.
"""

executive_note = """
Headline: the scoring rule that puts the cut at x0+x1=3 is clearly better than the cut at 4
(average loss 0.37 vs 0.50 on the same six labelled cases).
Why it matters: a rule that is confidently wrong is penalised hard, which is what we want in
admission, credit, or medical flagging. Recommend fitting the cut by minimising this loss on
a larger labelled set before any rollout.
"""

nonspecialist_note = """
Think of a simple size rule for a tumour: small ones are usually benign, large ones usually not.
We score a rule by how 'surprised' it is when it is wrong. A rule that says 'almost certainly
malignant' about a tumour that was benign is treated as a big mistake. On our tiny sketch,
the tighter size cut is less surprised overall than the looser one, so we keep the tighter cut
until we see more cases.
"""

print(analyst_note)
print(executive_note)
print(nonspecialist_note)



## 8. Flowchart + saved figures


In [ ]:
from IPython.display import Image, display
import os
for p in ["logistic_loss_cost_flowchart.png",
          "logistic_loss_curves.png",
          "logistic_vs_squared_heatmap.png",
          "logistic_vs_squared_surface.png",
          "logistic_cost_boundaries.png",
          "logistic_cost_simulation.png",
          "logistic_loss_1d_example.png"]:
    if os.path.exists(p):
        display(Image(p, width=620))
    else:
        print("missing", p)



## Key takeaways
1. Squared error + sigmoid is the wrong objective for 0/1 labels.
2. Logistic loss is two log curves glued by $y$; cost is their average.
3. $J$ is convex in $(w,b)$ for linear logistic regression — GD can find the global min.
4. Clip probabilities; one `log(0)` wrecks a whole training step.
5. Cost is a *comparison tool* even before you train: lower $J$ means the current $(w,b)$ is more consistent with the labels.
